# Phase 1 — Neurone unique, forward pass et calcul d'erreur

## Objectif
Coder un neurone unique capable de prédire si un point 2D appartient à la classe 0 ou 1.

Dans cette phase, on ne fait **pas encore d'entraînement**.
On fixe les poids manuellement, puis on observe :

- la somme pondérée \( z = X @ w + b \)
- la sortie après sigmoid
- la Binary Cross-Entropy (BCE)

## Ce que je veux comprendre
- ce qu'est un poids
- ce qu'est un biais
- ce qu'est un forward pass
- comment calculer une probabilité avec sigmoid
- comment mesurer l'erreur avec la BCE

In [2]:
import numpy as np

## 1. Dataset jouet

On utilise 4 points 2D avec des labels binaires.
L'idée est simplement d'avoir un petit problème très simple pour observer les calculs.

In [3]:
X = np.array([
    [0.2, 0.1],
    [0.8, 0.9],
    [0.3, 0.7],
    [0.9, 0.2],
])

y = np.array([0, 1, 1, 0])

print("X shape :", X.shape)
print("y shape :", y.shape)
print()
print("X =")
print(X)
print()
print("y =", y)

X shape : (4, 2)
y shape : (4,)

X =
[[0.2 0.1]
 [0.8 0.9]
 [0.3 0.7]
 [0.9 0.2]]

y = [0 1 1 0]


## 2. Fonction sigmoid

La sigmoid transforme une valeur réelle en probabilité entre 0 et 1.

Formule :

\[
\sigma(x) = \frac{1}{1 + e^{-x}}
\]

On l'utilise ici parce qu'on fait une classification binaire.

In [4]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

## 3. Forward pass

Le forward pass d'un neurone consiste à :

1. calculer la somme pondérée :
   \[
   z = X @ w + b
   \]

2. appliquer la sigmoid :
   \[
   \hat{y} = \sigma(z)
   \]

Ici :
- `X` contient les points d'entrée,
- `w` contient les poids,
- `b` est le biais.

In [5]:
def forward(X, w, b):
    z = np.dot(X, w) + b
    return sigmoid(z)

## 4. Fonction de coût : Binary Cross-Entropy

Comme on est en classification binaire, on utilise la BCE :

\[
\text{BCE} = -\frac{1}{n} \sum \left( y \log(\hat{y}) + (1-y)\log(1-\hat{y}) \right)
\]

On utilise `np.clip` pour éviter les problèmes numériques avec `log(0)`.

In [6]:
def compute_loss(y_true, y_pred):
    y_pred = np.clip(y_pred, 1e-7, 1 - 1e-7)
    return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))

## 5. Poids fixés manuellement

Dans cette phase, on ne fait pas encore d'entraînement.
On choisit simplement :
- `w = [0.5, -0.3]`
- `b = 0.1`

Puis on calcule les prédictions et la loss.

In [7]:
w = np.array([0.5, -0.3])
b = 0.1

print("w =", w)
print("b =", b)

w = [ 0.5 -0.3]
b = 0.1


In [8]:
y_pred = forward(X, w, b)
loss = compute_loss(y, y_pred)

print("Prédictions :", y_pred.round(3))
print("Étiquettes  :", y)
print("Loss BCE    :", round(loss, 4))

Prédictions : [0.542 0.557 0.51  0.62 ]
Étiquettes  : [0 1 1 0]
Loss BCE    : 0.7519


## 6. Interprétation

- Si une prédiction est proche de 1 pour une vraie étiquette 1, c'est bien.
- Si une prédiction est proche de 0 pour une vraie étiquette 0, c'est bien.
- Si les prédictions sont mauvaises ou hésitantes, la loss augmente.

Dans cette phase, les poids sont arbitraires.
Donc on n'attend pas forcément de bonnes prédictions : on vérifie surtout que le calcul fonctionne.

## 7. Test qualité — cas limite : entrées nulles

Si toutes les entrées valent 0, alors :

\[
z = X @ w + b = b
\]

La prédiction dépend donc uniquement du biais.
On doit obtenir la même valeur `sigmoid(b)` pour tous les exemples.

In [9]:
X_zero = np.zeros((4, 2))
y_pred_zero = forward(X_zero, w, b)

print("X_zero =")
print(X_zero)
print()
print("Predictions avec X=0 :", y_pred_zero.round(3))
print("Valeur attendue      :", round(sigmoid(b), 3))

X_zero =
[[0. 0.]
 [0. 0.]
 [0. 0.]
 [0. 0.]]

Predictions avec X=0 : [0.525 0.525 0.525 0.525]
Valeur attendue      : 0.525


## 8. Test qualité — scénario adversarial : poids nuls

Si :
- `w = [0, 0]`
- `b = 0`

alors :
- \( z = 0 \)
- \( \sigma(0) = 0.5 \)

Toutes les prédictions valent donc 0.5.
Le support indique que la BCE doit alors être proche de :

\[
-\log(0.5) \approx 0.693
\]

In [10]:
w_bad = np.zeros(2)
b_bad = 0.0

y_pred_bad = forward(X, w_bad, b_bad)
loss_bad = compute_loss(y, y_pred_bad)

print("w_bad =", w_bad)
print("b_bad =", b_bad)
print("Prédictions adversariales :", y_pred_bad.round(3))
print("Loss adversariale         :", round(loss_bad, 4))
print("Valeur théorique attendue :", round(-np.log(0.5), 4))

w_bad = [0. 0.]
b_bad = 0.0
Prédictions adversariales : [0.5 0.5 0.5 0.5]
Loss adversariale         : 0.6931
Valeur théorique attendue : 0.6931


## 9. Conclusion de la phase 1

À ce stade, le neurone sait :
- prendre des données d'entrée,
- calculer une somme pondérée,
- produire une probabilité via sigmoid,
- mesurer son erreur avec la BCE.

Il ne sait **pas encore apprendre**.
La phase 2 consistera à ajouter :
- les gradients,
- la descente de gradient,
- la boucle d'entraînement.